# 02 - TiDE-RIN Model Training

This notebook demonstrates training the TiDE-RIN forecasting model
for hourly PED admission prediction.

The model is configured via `configs/model_config.yaml` and uses the
Darts library for time-series forecasting.

**Requirements:** GPU recommended for training (16 encoder/decoder layers).

In [ ]:
import sys
sys.path.insert(0, '../src')

import yaml
import torch
import random
import numpy as np
from preprocessing.data_pipeline import DataPipeline

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

## 1. Load Configuration

In [ ]:
with open('../configs/model_config.yaml', 'r') as f:
    config = yaml.safe_load(f)

model_cfg = config['model']
train_cfg = config['training']
print(f"Model: {model_cfg['name']}")
print(f"Input chunk: {model_cfg['input_chunk_length']}h, Output chunk: {model_cfg['output_chunk_length']}h")

## 2. Reproducibility

In [ ]:
SEED = train_cfg['random_state']
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
np.random.seed(SEED)
random.seed(SEED)
torch.set_float32_matmul_precision('high')

## 3. Prepare Data

In [ ]:
DATA_PATH = "../data/hourly_admissions.csv"  # <-- Replace with your data path

pipeline = DataPipeline(data_path=DATA_PATH)
raw_series, scaled_train, train_covs = pipeline.prepare()
print(f"Training series length: {len(scaled_train)}")

## 4. Initialize Model

In [ ]:
from pytorch_lightning.callbacks.early_stopping import EarlyStopping
from darts.models import TiDEModel

es_cfg = train_cfg['early_stopping']
stopper = EarlyStopping(
    monitor=es_cfg['monitor'],
    patience=es_cfg['patience'],
    min_delta=es_cfg['min_delta'],
    mode=es_cfg['mode'],
)

pl_trainer_kwargs = {
    "callbacks": [stopper],
    "accelerator": train_cfg.get('accelerator', 'auto'),
}

model = TiDEModel(
    input_chunk_length=model_cfg['input_chunk_length'],
    output_chunk_length=model_cfg['output_chunk_length'],
    num_encoder_layers=model_cfg['num_encoder_layers'],
    num_decoder_layers=model_cfg['num_decoder_layers'],
    hidden_size=model_cfg['hidden_size'],
    temporal_decoder_hidden=model_cfg['temporal_decoder_hidden'],
    use_layer_norm=model_cfg['use_layer_norm'],
    use_reversible_instance_norm=model_cfg['use_reversible_instance_norm'],
    dropout=model_cfg['dropout'],
    optimizer_kwargs={'lr': train_cfg['optimizer']['lr']},
    batch_size=train_cfg['batch_size'],
    n_epochs=train_cfg['n_epochs'],
    lr_scheduler_cls=torch.optim.lr_scheduler.ExponentialLR,
    lr_scheduler_kwargs={'gamma': train_cfg['lr_scheduler']['gamma']},
    pl_trainer_kwargs=pl_trainer_kwargs,
    random_state=SEED,
    save_checkpoints=True,
    model_name='TiDE_RIN_SOSAFED',
)

print("Model initialized.")

## 5. Train

In [ ]:
model.fit(
    series=scaled_train,
    past_covariates=train_covs,
    verbose=True,
)
print("Training complete.")

## 6. Save Model

In [ ]:
MODEL_SAVE_PATH = "../models/TiDE_RIN_latest"
model.save(MODEL_SAVE_PATH)
print(f"Model saved to {MODEL_SAVE_PATH}")